# Evaluating AI Agents — Course Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enprompta/worldcup2026/blob/main/notebooks/evaluating-ai-agents.ipynb)

**Presented by [Enprompta](https://enprompta.com).** One notebook for the whole course. You'll take a real, open agent — the **2026 World Cup Assistant** — trace it, and evaluate it across the three surfaces every agent has:

- **Router** — did it *decide* right? (search a live question vs. answer from memory; redirect off-topic)
- **Skill** — did each capability *deliver*? (relevant sources; every claim backed by a **citation**)
- **Path** — did it get there *efficiently*? (how many web searches; cost; latency)

It builds on the app's [`enprompta_quickstart.ipynb`](https://github.com/Enprompta/worldcup2026) and grows it into a full evaluation suite: trace → build & validate evaluators → router/skill/path → dataset & experiment → monitoring → capstone.

> **How to use this with the videos:** each Part matches the course modules noted in its heading. Run the cells top to bottom.

## What you need

- An **Anthropic API key** (`sk-ant-...`) — the agent calls Claude.
- Your **Enprompta API key** and **OTLP endpoint** (Project → Settings → Ingest).

In Colab, add these in the **🔑 Secrets** panel as `ANTHROPIC_API_KEY`, `ENPROMPTA_API_KEY`, and `ENPROMPTA_OTLP_ENDPOINT`. The setup cell falls back to a prompt if a secret isn't set. Everything here fits the Enprompta **Free** plan.

---
## Part A — Trace the agent  ·  *Modules 1–2*

Before you can evaluate an agent you have to *see* it. We clone the uninstrumented app, wire OpenTelemetry to Enprompta, and turn every Claude call into a trace.

### A1 — Install dependencies
Vendor-neutral: Anthropic + OpenTelemetry + the OpenInference auto-instrumentor. Enprompta is just an OTLP endpoint, so no proprietary client is required.

In [ ]:
!pip -q install \
    "anthropic>=0.116.0" \
    openinference-instrumentation-anthropic \
    opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-http
print("installed")

### A2 — Keys & endpoint
Pulls from Colab Secrets first, then env vars, then prompts. Nothing is printed.

In [ ]:
import os
from getpass import getpass

def secret(name, prompt):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or getpass(prompt)

os.environ["ANTHROPIC_API_KEY"] = secret("ANTHROPIC_API_KEY", "Anthropic API key: ")
ENPROMPTA_API_KEY       = secret("ENPROMPTA_API_KEY", "Enprompta API key: ")
ENPROMPTA_OTLP_ENDPOINT = secret("ENPROMPTA_OTLP_ENDPOINT", "Enprompta OTLP endpoint: ")
PROJECT_NAME = "evaluating-ai-agents"
print("keys loaded")

### A3 — Get the app code
We import the app's real call-site config (`MODEL`, `SYSTEM_PROMPT`, `TOOLS`, `extract_citations`) so the notebook stays in sync with the agent.

In [ ]:
import sys, os
if not os.path.exists("worldcup2026"):
    !git clone -q https://github.com/enprompta/worldcup2026.git
sys.path.insert(0, "worldcup2026")

from worldcup import MODEL, SYSTEM_PROMPT, TOOLS, ALL_TOOLS, extract_citations
from worldcup_tools import CLIENT_TOOLS, TOOL_IMPLS
print("agent model:", MODEL, "| tools available:", 1 + len(CLIENT_TOOLS))

### A4 — Tracing (the foundation)
Wire OpenTelemetry to your Enprompta project and turn on the Anthropic auto-instrumentor. From here every Claude call becomes a span — prompt, completion, tokens, latency, tool use — shipped to Enprompta with no change to the app.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from openinference.instrumentation.anthropic import AnthropicInstrumentor

provider = TracerProvider(resource=Resource.create({"service.name": PROJECT_NAME}))
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(
    endpoint=ENPROMPTA_OTLP_ENDPOINT,
    headers={"Authorization": "Bearer " + ENPROMPTA_API_KEY},
)))
trace.set_tracer_provider(provider)
AnthropicInstrumentor().instrument(tracer_provider=provider)
print("tracing ON ->", ENPROMPTA_OTLP_ENDPOINT)

### A5 — A traced `ask()` that records what we'll evaluate
Same model, tools, and citation handling as the app, but it also returns the signals the three surfaces need: how many web searches ran, latency, and token usage. Every call here is auto-traced into Enprompta.

In [ ]:
import time, json
import anthropic
from opentelemetry import trace

client = anthropic.Anthropic()
tracer = trace.get_tracer("worldcup-agent")


def _run_client_tool(name, tool_input):
    """Execute one client tool inside a typed span. This is the instrumentation
    the course adds — the tools in worldcup_tools.py stay uninstrumented."""
    if name == "search_knowledge_base":
        # Show the deep, typed RAG sub-tree a vector store would emit.
        with tracer.start_as_current_span("retrieve_knowledge_base") as s:
            s.set_attribute("openinference.span.kind", "RETRIEVER")
            s.set_attribute("retrieval.query", tool_input.get("query", ""))
            with tracer.start_as_current_span("embed_query") as e:
                e.set_attribute("openinference.span.kind", "EMBEDDING")
                e.set_attribute("embedding.text", tool_input.get("query", ""))
            out = TOOL_IMPLS[name](**tool_input)
            with tracer.start_as_current_span("rerank_nodes") as r:
                r.set_attribute("openinference.span.kind", "RERANKER")
                r.set_attribute("reranker.top_k", out.get("count", 0))
            s.set_attribute("retrieval.documents.count", out.get("count", 0))
            return out
    with tracer.start_as_current_span(f"tool.{name}") as s:
        s.set_attribute("openinference.span.kind", "TOOL")
        s.set_attribute("tool.name", name)
        s.set_attribute("tool.parameters", json.dumps(tool_input))
        out = TOOL_IMPLS[name](**tool_input)
        s.set_attribute("tool.output", json.dumps(out, default=str)[:1500])
        return out


def ask(question, history=None, system=None, tools=TOOLS):
    """Run one turn as a traced agent. Returns a rich run dict.

    `tools` defaults to web_search only (Parts B-D evaluate that agent). Pass
    `tools=ALL_TOOLS` to let it also call the client-side tools; each tool call
    becomes its own typed span nested under one AGENT span.
    """
    messages = (history or []) + [{"role": "user", "content": question}]
    system = system or SYSTEM_PROMPT
    blocks, citations, tools_used = [], [], []
    in_tok = out_tok = 0
    t0 = time.time()
    with tracer.start_as_current_span("agent.worldcup") as root:
        root.set_attribute("openinference.span.kind", "AGENT")
        root.set_attribute("input.value", question)
        for _ in range(8):  # cap web_search pauses + client tool rounds
            resp = client.messages.create(
                model=MODEL, max_tokens=4096, system=system, tools=tools, messages=messages,
            )
            messages.append({"role": "assistant", "content": resp.content})
            blocks += resp.content
            citations += extract_citations(resp.content)
            u = getattr(resp, "usage", None)
            if u:
                in_tok += getattr(u, "input_tokens", 0) or 0
                out_tok += getattr(u, "output_tokens", 0) or 0
            if resp.stop_reason == "pause_turn":        # web_search runs server-side
                messages.append({"role": "user", "content": "Please continue."})
                continue
            if resp.stop_reason == "tool_use":           # client tools requested
                results = []
                for b in resp.content:
                    if getattr(b, "type", None) == "tool_use" and b.name in TOOL_IMPLS:
                        tools_used.append(b.name)
                        out = _run_client_tool(b.name, b.input or {})
                        results.append({"type": "tool_result", "tool_use_id": b.id,
                                        "content": json.dumps(out, default=str)})
                if results:
                    messages.append({"role": "user", "content": results})
                    continue
            break
        text = "".join(b.text for b in blocks if getattr(b, "type", None) == "text")
        root.set_attribute("output.value", text[:1500])
    n_searches = sum(
        1 for b in blocks
        if getattr(b, "type", None) == "server_tool_use" and getattr(b, "name", None) == "web_search"
    )
    seen, cites = set(), []
    for c in citations:
        if c["url"] not in seen:
            seen.add(c["url"]); cites.append(c)
    return {
        "question": question, "answer": text, "citations": cites,
        "n_searches": n_searches, "n_tool_calls": len(tools_used), "tools_used": tools_used,
        "latency_s": round(time.time() - t0, 2),
        "input_tokens": in_tok, "output_tokens": out_tok,
    }

print("ask() ready — traced agent (web_search by default; tools=ALL_TOOLS for the full toolset)")

### A6 — Produce a few runs (including a failing one)
We ask a static-fact question, a live question, and the classic trap — *"who won the 2026 final?"* — which the agent may answer from memory without searching. Open **Observability** in Enprompta afterward: you'll see these as traces.

In [ ]:
runs = []
for q in [
    "When and where does the 2026 World Cup start?",     # static fact — no search needed
    "What are the current Group A standings?",            # live — needs a search
    "Who won the 2026 World Cup final?",                  # the trap — may answer from memory
]:
    r = ask(q)
    runs.append(r)
    print(f"\nQ: {q}")
    print(f"   searches={r['n_searches']}  citations={len(r['citations'])}  latency={r['latency_s']}s")
    print("  ", r["answer"][:160].replace("\n", " "), "...")

provider.force_flush()  # push traces now so they appear in the dashboard
print("\n-> Open Enprompta Observability to see these 3 traces.")

### A7 — A richer agentic trace (more tools -> a deeper tree)

The same `ask()` can run the agent with its **full toolset** (`tools=ALL_TOOLS`) — web search *plus* the client-side tools (standings, fixtures, qualification, timezone, and a knowledge base). A question that composes several of them produces a deep, **typed** span tree — `AGENT -> LLM -> TOOL ... -> RETRIEVER -> EMBEDDING/RERANKER -> LLM` — instead of one flat call. That extra depth is exactly the surface Parts B-D evaluate: the router's tool choices, each skill's output, and the path's cost.

In [ ]:
rich = ask(
    "Can Poland still qualify from Group A, and when does their final group game "
    "kick off in Los Angeles time? What's the tie-breaker if they finish level on points?",
    tools=ALL_TOOLS,
)
provider.force_flush()  # push the trace now

print("tools used:      ", rich["tools_used"])
print("web searches:    ", rich["n_searches"], " | client tool calls:", rich["n_tool_calls"])
print("latency:         ", rich["latency_s"], "s")
print("\n", rich["answer"][:500], "...")
print("\n-> Open Observability: one AGENT span over the LLM turns, the tool calls, "
      "and a RETRIEVER -> EMBEDDING/RERANKER sub-tree.")

---
## Part B — Build & validate evaluators  ·  *Modules 3–4*

An evaluation is a function: run in, verdict out. **Code-based** checks are cheap, strict, and deterministic — use them wherever a rule can express the check. Reach for an **LLM-as-judge** only where language judgment is genuinely needed. Then *validate the judge against human labels* — an unvalidated judge just hides uncertainty.

### B1 — Code-based evaluators (free, deterministic)
Three rules over a run: did it cite a source, did it actually search, and did it redirect an off-topic question.

In [ ]:
import re

REDIRECT = re.compile(r"(outside (my|its) focus|only (help|answer).*world cup|not about the 2026)", re.I)

def has_citation(run):      # grounding proxy: a claim with a source
    return len(run["citations"]) > 0

def used_search(run):
    return run["n_searches"] > 0

def redirected(run):        # for off-topic questions
    return bool(REDIRECT.search(run["answer"]))

for r in runs:
    print(f"{r['question'][:42]:44} has_citation={has_citation(r)!s:5} used_search={used_search(r)!s:5}")

### B2 — LLM-as-judge: groundedness (structured verdict)
The important question a rule can't answer: is *every factual claim* in the answer supported by a cited source? We force **structured JSON** so verdicts aggregate and failures are debuggable. A smaller/cheaper judge model is usually fine for a tight rubric.

In [ ]:
import json

JUDGE_MODEL = MODEL   # tip: a cheaper model (e.g. Haiku) is usually enough for a tight rubric

def judge_groundedness(run):
    src = "\n".join(f"- {c['title']}: {c['url']}" for c in run["citations"]) or "(no sources cited)"
    prompt = (
        f"Question: {run['question']}\n\nAnswer: {run['answer']}\n\nCited sources:\n{src}\n\n"
        "Is every factual claim in the Answer supported by a cited source? Ignore general "
        "correctness — only grounding in THESE sources. A plausible-but-uncited claim is NOT grounded.\n"
        'Respond ONLY as JSON: {"grounded": true or false, "unsupported_claims": [..], "reason": "one sentence"}'
    )
    r = client.messages.create(
        model=JUDGE_MODEL, max_tokens=300,
        system="You are a strict grounding evaluator for a sports assistant.",
        messages=[{"role": "user", "content": prompt}],
    )
    t = "".join(b.text for b in r.content if getattr(b, "type", None) == "text")
    m = re.search(r"\{.*\}", t, re.S)
    try:
        return json.loads(m.group(0)) if m else {"grounded": None, "reason": t[:120]}
    except Exception:
        return {"grounded": None, "reason": t[:120]}

for r in runs:
    v = judge_groundedness(r)
    print(f"{r['question'][:42]:44} grounded={v.get('grounded')!s:6} {v.get('reason','')[:70]}")

### B3 — Trust the judge: align it to human labels
Twenty careful human labels beat a thousand unchecked judge calls. We hand-label a small set, run the judge, and measure agreement with **precision / recall / F1** — treating **"ungrounded"** (the failure we want to catch) as the positive class. Below, edit `HUMAN` to your own judgement after reading each answer.

In [ ]:
# Small labelled set. grounded=True means "a human judged every claim supported".
LABELLED = [
    "When and where does the 2026 World Cup start?",
    "Which cities host the 2026 semi-finals?",
    "Who won the 2026 World Cup final?",
    "How many teams play in the 2026 World Cup?",
    "What is the current top scorer of the tournament?",
]
# TODO: run each, read the answer, and set your honest human label (True = grounded).
HUMAN = {q: None for q in LABELLED}   # e.g. HUMAN[LABELLED[2]] = False

def prf(labels, preds):
    # positive class = UNGROUNDED (not grounded)
    tp = sum(1 for l, p in zip(labels, preds) if (l is False) and (p is False))
    fp = sum(1 for l, p in zip(labels, preds) if (l is True)  and (p is False))
    fn = sum(1 for l, p in zip(labels, preds) if (l is False) and (p is True))
    tn = sum(1 for l, p in zip(labels, preds) if (l is True)  and (p is True))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": round(prec, 2), "recall": round(rec, 2), "f1": round(f1, 2)}

labels, preds = [], []
for q in LABELLED:
    r = ask(q)
    v = judge_groundedness(r)
    print(f"{q[:44]:46} judge_grounded={v.get('grounded')} | your HUMAN label={HUMAN[q]}")
    if HUMAN[q] is not None:
        labels.append(HUMAN[q]); preds.append(bool(v.get("grounded")))

if labels:
    print("\nJudge vs. humans:", prf(labels, preds))
else:
    print("\nSet a few HUMAN labels above, then re-run to get precision/recall/F1.")

---
## Part C — Router · Skill · Path  ·  *Modules 5–7*

Now we evaluate each of the three surfaces on its own terms.

### C1 — Router: search when needed, redirect off-topic
Two router failures: answering a live question **from memory** (no search) and answering an **off-topic** question instead of redirecting. We score against a small labelled set.

In [ ]:
ROUTER_SET = [
    {"q": "What are the current Group A standings?",         "needs_search": True,  "on_topic": True},
    {"q": "Who won the 2026 World Cup final?",               "needs_search": True,  "on_topic": True},
    {"q": "How many teams play in the 2026 World Cup?",      "needs_search": False, "on_topic": True},
    {"q": "Who is the best player in the NBA right now?",    "needs_search": False, "on_topic": False},
    {"q": "What's a good pasta recipe?",                     "needs_search": False, "on_topic": False},
]

rows = []
for item in ROUTER_SET:
    r = ask(item["q"])
    if item["on_topic"]:
        ok = (r["n_searches"] > 0) == item["needs_search"]     # searched iff it should have
        kind = "search_decision"
    else:
        ok = redirected(r) and r["n_searches"] == 0            # should redirect, not search
        kind = "off_topic_redirect"
    rows.append((item["q"], kind, ok, r["n_searches"]))
    print(f"{'PASS' if ok else 'FAIL':4} [{kind:18}] searches={r['n_searches']}  {item['q'][:40]}")

rate = sum(1 for _, _, ok, _ in rows if ok) / len(rows)
print(f"\nrouter score: {rate:.0%}")

### C2 — Skill: relevance + groundedness of the search skill
Web search is our retrieval skill. Two questions, evaluated separately: were the **sources relevant** to the question, and is the **answer grounded** in them? Low relevance + high groundedness = faithful use of bad sources (fix the search); high relevance + low groundedness = it had good sources and ignored them (fix the prompt).

In [ ]:
def judge_relevance(run):
    src = "\n".join(f"- {c['title']}: {c['url']}" for c in run["citations"]) or "(none)"
    prompt = (f"Question: {run['question']}\n\nSources found:\n{src}\n\n"
              "Do these sources contain information needed to answer the question? "
              'Respond ONLY as JSON: {"relevant": true or false, "reason": "one sentence"}')
    r = client.messages.create(model=JUDGE_MODEL, max_tokens=200,
                               system="You grade retrieval relevance.",
                               messages=[{"role": "user", "content": prompt}])
    t = "".join(b.text for b in r.content if getattr(b, "type", None) == "text")
    m = re.search(r"\{.*\}", t, re.S)
    try:
        return json.loads(m.group(0)) if m else {"relevant": None}
    except Exception:
        return {"relevant": None}

for q in ["What are the current Group A standings?", "Which cities host the 2026 final?"]:
    r = ask(q)
    rel = judge_relevance(r); gnd = judge_groundedness(r)
    print(f"{q[:40]:42} relevant={rel.get('relevant')!s:6} grounded={gnd.get('grounded')!s:6} "
          f"has_citation={has_citation(r)}")

### C3 — Path: how many searches, and what did it cost?
An agent that searches five times for a one-search question is *worse* even when the answer is right — slower, pricier, more fragile. We score search count against an expected budget and estimate cost. (Rates below are illustrative — set yours.)

In [ ]:
PRICE_IN, PRICE_OUT = 3.0 / 1e6, 15.0 / 1e6   # $/token (illustrative)
PRICE_SEARCH = 0.01                            # $/web_search (illustrative)

def est_cost(run):
    return round(run["input_tokens"] * PRICE_IN + run["output_tokens"] * PRICE_OUT
                 + run["n_searches"] * PRICE_SEARCH, 4)

PATH_SET = [
    ("How many teams play in the 2026 World Cup?", 0),   # static — expect 0 searches
    ("What are the current Group A standings?",    1),   # one search should do it
]
for q, budget in PATH_SET:
    r = ask(q)
    over = r["n_searches"] > budget
    print(f"{'OVER-SEARCH' if over else 'ok':11} searches={r['n_searches']} (budget {budget}) "
          f"latency={r['latency_s']}s  ~${est_cost(r)}  {q[:34]}")

---
## Part D — Dataset, experiment, monitoring  ·  *Modules 8–9*

Measuring what *is* only pays off when you can **prove a change is better**. We freeze a dataset, run two prompt versions over it, and diff the scores — then look at taking evals online.

### D1 — Experiment: prompt v1 vs. v2 on a frozen dataset
`v2` adds a strict rule: always search for any live fact, and never state a claim you can't cite. Same dataset, only the prompt changes — so any score difference is caused by the change.

In [ ]:
DATASET = [
    "What are the current Group A standings?",
    "Who won the 2026 World Cup final?",
    "Which cities host the 2026 semi-finals?",
    "What is the current top scorer of the tournament?",
    "How many teams play in the 2026 World Cup?",
]

PROMPT_V2 = SYSTEM_PROMPT + (
    "\n\nSTRICT RULES: For any result, standing, fixture, score, or record you MUST use the "
    "web_search tool before answering. Every factual claim MUST be supported by a citation. "
    "If you cannot find a source, say you don't know rather than guessing."
)

def run_suite(dataset, system, label):
    grounded = cited = 0
    total_searches = 0
    for q in dataset:
        r = ask(q, system=system)
        cited += 1 if has_citation(r) else 0
        total_searches += r["n_searches"]
        v = judge_groundedness(r)
        grounded += 1 if v.get("grounded") else 0
    n = len(dataset)
    m = {"grounded_rate": round(grounded / n, 2), "citation_rate": round(cited / n, 2),
         "avg_searches": round(total_searches / n, 2)}
    print(f"[{label}] {m}")
    return m

v1 = run_suite(DATASET, SYSTEM_PROMPT, "v1-baseline")
v2 = run_suite(DATASET, PROMPT_V2,    "v2-strict")
print("\nDIFF  grounded:", round(v2['grounded_rate'] - v1['grounded_rate'], 2),
      " citation:", round(v2['citation_rate'] - v1['citation_rate'], 2),
      " avg_searches:", round(v2['avg_searches'] - v1['avg_searches'], 2))

### D2 — From offline to online (CI gate + monitoring)
The same suite becomes a **CI gate** (fail a build if grounding drops below a baseline) and, in Enprompta, an **online monitor** on sampled live traffic with an **alert**. Below is the gate shape; the continuous/alerting side lives in the Enprompta app (Evaluations → run continuously; Alerts).

In [ ]:
BASELINE_GROUNDED = 0.60   # your agreed floor

def ci_gate(dataset, system):
    m = run_suite(dataset, system, "ci-check")
    passed = m["grounded_rate"] >= BASELINE_GROUNDED
    print("CI GATE:", "PASS ✅" if passed else "FAIL ❌ — grounding below baseline")
    return passed

ci_gate(DATASET, PROMPT_V2)
# In practice: run this in GitHub Actions on every PR; block merge on FAIL.
# For live traffic: enable these evaluators to run continuously (sampled) in Enprompta + set an alert.

---
## Part E — Capstone  ·  *Module 10*

You've built the whole loop. For your certificate, assemble it for this agent (or one of your own) and share it from Enprompta.

**Submit a project that has:**
1. A **versioned dataset** (20+ questions, including known failures like the "who won the final?" trap).
2. Evaluators for **all three surfaces** — a router check, a skill check with a **validated** judge (report its F1), and a path check.
3. An **experiment** comparing prompt v1 → v2 with an attributable improvement *and* one acknowledged regression/limitation.
4. One **online monitor** (continuous, sampled) with an **alert**.
5. A short **write-up**: a named failure → the fix → the number that moved.

**Share it:** open your Enprompta project, share it read-only, and submit with your write-up to earn the **Enprompta — Agent Evaluation** certificate.

---
### Where to go next
- Point this loop at **your own agent** — instrument one real call site and find one real bug.
- Log eval scores **back onto the traces** via the Enprompta evals API so verdicts sit beside each span.
- Read the guides: [Tracing](https://enprompta.com/docs/tracing) · [Evaluations](https://enprompta.com/docs/workbench) · [Getting started](https://enprompta.com/docs/getting-started).

*Evaluation isn't a phase at the end — it's the instrument panel you fly the whole way.*